# 01 — Flashbots Dataset Audit

**Purpose:** establish exactly what the Flashbots Mempool Dumpster data are before anything is modelled:
- **Identity:** the files are the registered ones (checksums).
- **Schema:** the 18 columns, their types and roles.
- **Quality:** missing values, duplicates, invalid timestamps, coverage gaps, internal consistency and outliers.

**Sources and their roles:**
- The raw ZIP archives are read **in place** and never extracted or modified. Their registered paths are in
  `data/manifests/flashbots_mempool_manifest.yaml`.
- The full row-by-row audit was produced by `Flashbots/data_audit/audit_raw_archives.py` and stored as
  `audit_report.json`. This notebook displays it.
- It also **re-verifies** the key facts on an independent path: archive SHA-256 recomputed here, and counts
  recomputed from the row-complete extract.
- Set `RUN_FULL_AUDIT = True` to recompute the entire audit in memory (about 90 s).

This notebook writes no files.

## Setup

In [1]:
from pathlib import Path
import sys, json

ROOT = Path.cwd()
while not (ROOT / "Flashbots" / "fbcommon.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
FB = ROOT / "Flashbots"
for sub in ("", "forecasting", "evaluation", "time_series", "data_audit", "preprocessing"):
    sys.path.insert(0, str(FB / sub))

import numpy as np
import pandas as pd
import fbcommon as fc
fc.plot_style()
%matplotlib inline
import matplotlib.pyplot as plt

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda v: f"{v:,.0f}" if abs(v) >= 1e4 else f"{v:.4g}")
print("repository:", ROOT)
print({k: v for k, v in fc.environment().items() if k != "platform"})

repository: C:\OJT_SEM3\AI-Assisted-Permissioned-Blockchain-Architecture-


{'python': '3.14.3', 'numpy': '2.5.4', 'pandas': '3.0.6', 'scipy': '1.18.1', 'scikit_learn': '1.9.1', 'matplotlib': '3.11.2', 'pyyaml': '6.0.3'}


In [2]:
RUN_FULL_AUDIT = False   # True: recompute the full row-by-row audit in memory (~90 s) and compare it with the stored report
AUDIT = json.loads((FB / "data_audit" / "audit_report.json").read_text(encoding="utf-8"))
DAYS = {d["date_utc"]: d for d in AUDIT["days"]}
manifest = fc.load_manifest()
print("stored audit generated:", AUDIT["generated_utc"])

stored audit generated: 2026-11-16T06:39:22+00:00


## 1. File identity: checksums recomputed from the actual archives

In [3]:
identity = []
for date, entry in manifest.items():
    path = Path(entry["source_path"])
    sha = fc.sha256_file(path)
    identity.append({"date_utc": date, "archive": path.name, "bytes": path.stat().st_size,
                     "sha256_recomputed": sha[:16] + "...", "matches_manifest": sha == entry["sha256"].upper(),
                     "rows_registered": entry["rows"]})
identity = pd.DataFrame(identity)
display(identity)
assert identity["matches_manifest"].all(), "archive differs from the registered manifest"

,date_utc,archive,bytes,sha256_recomputed,matches_manifest,rows_registered
0,2026-09-21,2026-09-21.csv.zip,100594099,DFF1A04366B772EC...,True,1055934
1,2026-10-12,2026-10-12.csv.zip,87369211,15C8C63E0DBBEBFB...,True,913856


## 2. Schema: what one row is

The README states that each row is one unique transaction hash **received by Flashbots' mempool collectors**
(geth/Infura nodes, Alchemy, bloXroute, Chainbound, Eden, ...) on that UTC day, and **not already on-chain** when it
was received. Below are the header and the first three rows of the raw file, read directly from the ZIP.

In [4]:
first = next(fc.iter_raw_chunks(manifest["2026-09-21"]["source_path"], as_text=True, chunk_rows=3))
print("columns:", len(first.columns), "| identical to the registered schema:", list(first.columns) == fc.EXPECTED_COLUMNS)
display(first.T)
assert list(first.columns) == fc.EXPECTED_COLUMNS

columns: 18 | identical to the registered schema: True


,0,1,2
timestamp_ms,1789948800143,1789948800327,1789948800509
hash,0x8d78c893aac48ae81346d35cdac31a1d0d0a5b73a131...,0x4ee3e967794278fa1d856074e1486aec46cc27e68a99...,0x749986f0f02a761ca9f83660ee78f1b084d7fcd670ef...
chain_id,1,1,1
from,0x485a8294b94221ed85e343ea2832613d7ccca7e5,0xa690621b1d2f8cc06de2de11e4cf69935f82655a,0x37896c84010b4f648454c295c8c0fcb45cb72385
to,0xdac17f958d2ee523a2206206994597c13d831ec7,0xdac17f958d2ee523a2206206994597c13d831ec7,0xc89bdc996b8cf34779c8db47f0ca5fd6c1cca3e9
value,0,0,4348519700000
nonce,119,99368,22
gas,70000,160000,21000
gas_price,1103589748,200000000000,65437500
gas_tip_cap,1000000000,4786552,65437500


In [5]:
roles = pd.read_csv(FB / "data_audit" / "column_profile.csv")
roles = roles[roles.date_utc == "2026-09-21"][["column", "role", "inferred_type", "nulls", "null_share"]]
roles.merge(pd.read_csv(FB / "data_audit" / "column_profile.csv").query("date_utc == '2026-10-12'")[["column", "nulls"]],
            on="column", suffixes=("_21sep", "_12oct"))

,column,role,inferred_type,nulls_21sep,null_share,nulls_12oct
0,timestamp_ms,reception timestamp (arrival time) - forecasti...,integer,0,0,0
1,hash,identity / descriptive,text,0,0,0
2,chain_id,metadata,integer,0,0,0
3,from,identity / descriptive,text,0,0,0
4,to,identity / descriptive,text,410,0.0003883,741
5,value,transaction attribute (known at reception),decimal,0,0,0
6,nonce,identity / descriptive,integer,0,0,0
7,gas,transaction attribute (known at reception),integer,0,0,0
8,gas_price,transaction attribute (known at reception),integer,0,0,0
9,gas_tip_cap,transaction attribute (known at reception),integer,0,0,0


**Reading the roles:**
- `timestamp_ms` is the reception time and the only arrival time.
- The three `included_*` / `inclusion_delay_ms` columns are known only **after** inclusion, so they are never
  forecasting inputs (leakage).
- `sources` is empty in every row, so per-collector timing is unavailable.
- `to` is empty for contract creations.
- `data_4bytes` is empty for transactions without calldata (plain transfers).

## 3. Rows, duplicates, timestamps (stored full audit)

In [6]:
overview = pd.DataFrame({d: {
    "rows": r["rows"], "rows_match_manifest": r["rows_match_manifest"], "schema_ok": r["schema_matches_registered"],
    "duplicate_tx_hash_rows": r["duplicates"]["tx_hash_duplicate_rows"],
    "replacement_(sender,nonce)_pairs": r["duplicates"]["sender_nonce_pairs_with_multiple_hashes"],
    "first_utc": r["timestamps"]["first_utc"], "last_utc": r["timestamps"]["last_utc"],
    "timestamp_decreases": r["timestamps"]["decreases"], "exact_timestamp_ties": r["timestamps"]["exact_ties"],
    "seconds_with_zero_arrivals": r["timestamps"]["seconds_with_zero_arrivals"],
    "longest_zero_run_s": r["timestamps"]["longest_zero_arrival_run_s"],
    "max_interarrival_ms": r["timestamps"]["max_interarrival_ms"],
    "never_included_rows": r["inclusion"]["not_included_rows"]} for d, r in DAYS.items()})
display(overview)
print("combined rows:", AUDIT["combined"]["rows"], "| hashes present on both dates:", AUDIT["combined"]["tx_hashes_present_on_both_dates"])
assert AUDIT["combined"]["rows"] == 1_969_790 and AUDIT["combined"]["schemas_identical"]

,2026-09-21,2026-10-12
rows,1055934,913856
rows_match_manifest,True,True
schema_ok,True,True
duplicate_tx_hash_rows,0,0
"replacement_(sender,nonce)_pairs",4998,3057
first_utc,2026-09-21T00:00:00.143000+00:00,2026-10-12T00:00:00.031000+00:00
last_utc,2026-09-21T23:59:59.990000+00:00,2026-10-12T23:59:59.996000+00:00
timestamp_decreases,0,0
exact_timestamp_ties,46198,27802
seconds_with_zero_arrivals,41,83


combined rows: 1969790 | hashes present on both dates: 0


## 4. Internal consistency checks (18 per day)

In [7]:
checks = pd.read_csv(FB / "data_audit" / "consistency_checks.csv")
table = checks.pivot(index="check", columns="date_utc", values="failing_rows")
table["definition"] = checks.drop_duplicates("check").set_index("check")["definition"]
table

date_utc,2026-09-21,2026-10-12,definition
check,,,
chain_id_not_1,0,0,Ethereum mainnet only
columns_match_registered_schema,0,0,"header equals the 18 registered columns, in order"
duplicate_tx_hash_rows,0,0,rows whose hash already appeared
empty_calldata_rows,385560,364222,informational: plain transfers (no calldata)
gas_limit_above_block_limit_45M,0,0,cannot fit in one mainnet block (45M limit ass...
gas_limit_below_21000,0,0,intrinsic minimum gas for any tx
hash_format_0x_64hex,0,0,tx hash is 0x + 64 hex chars
inclusion_delay_formula_mismatch,0,0,inclusion_delay_ms == included_block_timestamp...
inclusion_delay_over_1h,303,239,landed more than 1 h after first seen


**Interpretation:**
- Every structural check passes:
  - schema, hash format, no duplicates, timestamps inside the day and in order;
  - mainnet only;
  - the inclusion-delay formula from the README holds for every included row;
  - tip cap <= fee cap;
  - no gas limit below 21,000 or above 45M;
  - selector present exactly when calldata >= 4 bytes.
- The non-zero rows are **properties of the data, not errors**: exact ties, never-included transactions, negative
  inclusion delays (block time earlier than first sighting, which the README allows) and plain transfers. All rows
  are kept.

## 5. Independent re-check from the row-complete extract

In [8]:
recheck = []
for date in fc.DATES:
    tx = fc.load_extract(date)
    ts = tx["timestamp_ms"]
    off = ts - fc.day_start_ms(date)
    gaps = np.diff(ts)
    per_s = np.bincount(off // 1000, minlength=fc.DAY_S)
    r = DAYS[date]
    recheck.append({"date_utc": date, "rows": len(ts), "rows_audit": r["rows"],
                    "outside_day": int(((off < 0) | (off >= fc.DAY_MS)).sum()), "decreases": int((gaps < 0).sum()),
                    "exact_ties": int((gaps == 0).sum()), "exact_ties_audit": r["timestamps"]["exact_ties"],
                    "zero_seconds": int((per_s == 0).sum()), "zero_seconds_audit": r["timestamps"]["seconds_with_zero_arrivals"],
                    "never_included": int((tx["included_block_timestamp_ms"] == 0).sum()),
                    "never_included_audit": r["inclusion"]["not_included_rows"]})
recheck = pd.DataFrame(recheck)
display(recheck)
for a, b in [("rows", "rows_audit"), ("exact_ties", "exact_ties_audit"), ("zero_seconds", "zero_seconds_audit"), ("never_included", "never_included_audit")]:
    assert (recheck[a] == recheck[b]).all(), a
assert (recheck[["outside_day", "decreases"]] == 0).all().all()

,date_utc,rows,rows_audit,outside_day,decreases,exact_ties,exact_ties_audit,zero_seconds,zero_seconds_audit,never_included,never_included_audit
0,2026-09-21,1055934,1055934,0,0,46198,46198,41,41,16604,16604
1,2026-10-12,913856,913856,0,0,27802,27802,83,83,21149,21149


In [9]:
if RUN_FULL_AUDIT:
    import audit_raw_archives as ara
    for date in fc.DATES:
        report, _, consistency, _, _ = ara.audit_day(date, manifest[date])
        stored = checks[checks.date_utc == date].set_index("check")["failing_rows"]
        assert (consistency.set_index("check")["failing_rows"] == stored).all()
        print(date, "full audit recomputed: identical to the stored report")
else:
    print("RUN_FULL_AUDIT is False: using the stored full audit (generated", AUDIT["generated_utc"] + ")")

RUN_FULL_AUDIT is False: using the stored full audit (generated 2026-11-16T06:39:22+00:00)


## 6. Reconciliation with the earlier stored audit (`experiments/results/flashbots_archive_audit.json`)

In [10]:
recon = pd.DataFrame(AUDIT["reconciliation"])
recon["this_audit"] = recon["this_audit"].astype(str).str[:60]
recon["stored_audit"] = recon["stored_audit"].astype(str).str[:60]
display(recon)
assert recon["match"].all()

,date_utc,field,this_audit,stored_audit,match
0,2026-09-21,rows,1055934,1055934,True
1,2026-09-21,sha256,DFF1A04366B772EC8DE6708450849082F2AFD12DDFC2BA...,DFF1A04366B772EC8DE6708450849082F2AFD12DDFC2BA...,True
2,2026-09-21,duplicate_tx_hashes,0,0,True
3,2026-09-21,timestamp_decreases,0,0,True
4,2026-09-21,exact_tie_gaps,46198,46198,True
5,2026-09-21,null_counts,"{'timestamp_ms': 0, 'hash': 0, 'chain_id': 0, ...","{'timestamp_ms': 0, 'hash': 0, 'chain_id': 0, ...",True
6,2026-10-12,rows,913856,913856,True
7,2026-10-12,sha256,15C8C63E0DBBEBFB54BD9A2F0DBF3529E93BDDAA21C5FF...,15C8C63E0DBBEBFB54BD9A2F0DBF3529E93BDDAA21C5FF...,True
8,2026-10-12,duplicate_tx_hashes,0,0,True
9,2026-10-12,timestamp_decreases,0,0,True


## 7. Coverage gaps and outliers (counted, never removed)

In [11]:
gaps = pd.read_csv(FB / "data_audit" / "longest_gaps.csv")
display(gaps.groupby("date_utc").head(3))
display(pd.DataFrame({d: r["outliers_counted_not_removed"] for d, r in DAYS.items()}))

,date_utc,rank,gap_ms,from_utc,to_utc
0,2026-09-21,1,1961,2026-09-21T03:18:33.020000+00:00,2026-09-21T03:18:34.981000+00:00
1,2026-09-21,2,1779,2026-09-21T04:39:43.933000+00:00,2026-09-21T04:39:45.712000+00:00
2,2026-09-21,3,1757,2026-09-21T02:06:43.681000+00:00,2026-09-21T02:06:45.438000+00:00
10,2026-10-12,1,9363,2026-10-12T04:30:54.786000+00:00,2026-10-12T04:31:04.149000+00:00
11,2026-10-12,2,3150,2026-10-12T03:56:55.119000+00:00,2026-10-12T03:56:58.269000+00:00
12,2026-10-12,3,2443,2026-10-12T00:26:54.348000+00:00,2026-10-12T00:26:56.791000+00:00


,2026-09-21,2026-10-12
per_second_count_gt_p99_9,86,87
per_second_count_max,362,817
per_second_count_max_at_utc,2026-09-21T19:02:51+00:00,2026-10-12T05:22:25+00:00
gas_fee_cap_gwei_gt_1000,53,17
gas_fee_cap_gwei_max,4307,1507
gas_limit_ge_10M,1501,4477
data_size_gt_100kB,215,145
robust_z_gt_10_per_second_counts,116,168


## Conclusion

| Question | Answer |
|---|---|
| Are these the registered files? | Yes. SHA-256 recomputed here matches the manifest. |
| Size | 1,055,934 + 913,856 = **1,969,790** transactions, identical 18-column schema |
| Missing values | Only where meaningful: `to` (contract creations), `data_4bytes` (no calldata), `sources` (empty in all rows) |
| Duplicates | 0 duplicate hashes, within and across days. Replacement transactions (same sender and nonce) are kept as real arrivals |
| Timestamps | UTC epoch ms, all inside the file's day, in order. Exact ties kept |
| Gaps | Empty seconds are rare (41 / 83). The longest gap is 9.4 s (12 Oct 04:30 UTC); a collector outage is possible but **unresolved** |
| Outliers | Bursts up to 817 tx in one second; extreme fee bids. Counted, not removed |
| Suitable for | Arrival-rate time-series analysis. **Not** for multi-day seasonality, per-collector analysis or private orderflow |

Next: `../eda/02_flashbots_eda.ipynb`.